In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import LSTM, Dense, GRU
from tensorflow.keras.utils import to_categorical
import numpy as np
from sklearn.metrics import accuracy_score
import tensorflow as tf
from collections import defaultdict

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Loading our data and exploratory data analysis (EDA)

In [ ]:
import pandas as pd

# !!! IMPORTANT !!!
# !! YOU MUST UPDATE THIS PATH TO MATCH YOUR KAGGLE INPUT FOLDER !!
# For example: '/kaggle/input/cert-insider-threat-r42/email.csv'
data = pd.read_csv('/kaggle/input/cert-insider-threat/email.csv')

data['date'] = pd.to_datetime(data['date'], format='%m/%d/%Y %H:%M:%S')

missing_values = data.isnull().sum()

data_info = data.describe()
data_info = data.info()

email_volume = data.groupby(data['date'].dt.date).size()

user_frequency = data['user'].value_counts()

attachment_analysis = data['attachments'].value_counts()

print("Missing Values:\n", missing_values)

In [ ]:
print("Data Info:\n", data_info)

In [ ]:
print("Email Volume Over Time:\n", email_volume)

In [ ]:
print("User Frequency:\n", user_frequency)

In [ ]:
print("Attachment Analysis:\n", attachment_analysis)

# Feature Engineering & Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data['date'] = pd.to_datetime(data['date'], format='%m/%d/%Y %H:%M:%S')

data['cc'] = data['cc'].fillna('')
data['bcc'] = data['bcc'].fillna('')

data['num_recipients'] = data['to'].str.count(';') + data['cc'].str.count(';') + data['bcc'].str.count(';') + 1
data['hour'] = data['date'].dt.hour
data['day_of_week'] = data['date'].dt.dayofweek

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data['hour'], bins=24, kde=False)
plt.title('Email Activity by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Email Count')
plt.show()

In [ ]:
top_users = data['user'].value_counts().head(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_users.index, y=top_users.values)
plt.title('Top 10 Users by Email Activity')
plt.xlabel('User')
plt.ylabel('Email Count')
plt.show()

# Anomaly Detection (Isolation Forest - Full Data)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

features = ['size', 'attachments', 'num_recipients', 'hour', 'day_of_week']
X = data[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
data['anomaly'] = model.fit_predict(X_scaled)

anomalies = data[data['anomaly'] == -1]

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(anomalies['date'].dt.date, bins=30, kde=False)
plt.title('Anomalous Email Activity Over Time')
plt.xlabel('Date')
plt.ylabel('Anomalous Email Count')
plt.show()

print(f"Number of Anomalies Detected: {len(anomalies)}")

# Data Preprocessing for ML Models (10% Sample)

In [ ]:
data_sample = data.sample(frac=0.1, random_state=42)

In [ ]:
data_sample['date'] = pd.to_datetime(data_sample['date'], format='%m/%d/%Y %H:%M:%S')
data_sample['cc'] = data_sample['cc'].fillna('')
data_sample['bcc'] = data_sample['bcc'].fillna('')

data_sample['num_recipients'] = data_sample['to'].str.count(';') + data_sample['cc'].str.count(';') + data_sample['bcc'].str.count(';') + 1
data_sample['hour'] = data_sample['date'].dt.hour
data_sample['day_of_week'] = data_sample['date'].dt.dayofweek

In [ ]:
tfidf = TfidfVectorizer(max_features=1000)
content_tfidf = tfidf.fit_transform(data_sample['content']).toarray()
content_tfidf_df = pd.DataFrame(content_tfidf, columns=tfidf.get_feature_names_out())

In [ ]:
features = ['size', 'attachments', 'num_recipients', 'hour', 'day_of_week']
X_numeric = data_sample[features]
X = pd.concat([X_numeric.reset_index(drop=True), content_tfidf_df.reset_index(drop=True)], axis=1)

In [ ]:
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)
X_scaled = pd.concat([pd.DataFrame(X_numeric_scaled, columns=features).reset_index(drop=True), content_tfidf_df.reset_index(drop=True)], axis=1)

In [ ]:
model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
data_sample['anomaly'] = model.fit_predict(X_scaled)
data_sample['anomaly'] = data_sample['anomaly'].map({1: 0, -1: 1})

In [ ]:
y = data_sample['anomaly']
X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

X_train_users = data_sample.loc[y_train_df.index]['user']

X_train = X_train_df.values
X_test = X_test_df.values
y_train = y_train_df.values
y_test = y_test_df.values

# Model 1: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_df, y_train_df)
y_pred_rf = rf.predict(X_test_df)
print("Random Forest Classification Report:")
print(classification_report(y_test_df, y_pred_rf))

In [ ]:
acc = accuracy_score(y_test_df, y_pred_rf)
print(f"Accuracy using random forest is: {acc*100:.3f} %")

# Model 2: LSTM

In [ ]:
X_train_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

In [ ]:
lstm_model = Sequential()
lstm_model.add(LSTM(50, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2]), return_sequences=True))
lstm_model.add(LSTM(50))
lstm_model.add(Dense(2, activation='softmax'))

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.fit(X_train_lstm, y_train_cat, epochs=10, batch_size=64, validation_split=0.2, verbose=2)
y_pred_lstm = np.argmax(lstm_model.predict(X_test_lstm), axis=1)
print("LSTM Classification Report:")
print(classification_report(y_test, y_pred_lstm))

In [ ]:
acc = accuracy_score(y_test, y_pred_lstm)
print(f"The Accuracy using LSTM method is : {acc*100:.3f} %")

# Model 3: GRU (Centralized)

In [ ]:
X_train_gru = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_gru = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
y_train_cat_gru = to_categorical(y_train)
y_test_cat_gru = to_categorical(y_test)

gru_model = Sequential()
gru_model.add(GRU(50, input_shape=(X_train_gru.shape[1], X_train_gru.shape[2]), return_sequences=True))
gru_model.add(GRU(50))
gru_model.add(Dense(2, activation='softmax'))

gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print("--- Centralized GRU Training (for comparison) ---")
gru_model.fit(X_train_gru, y_train_cat_gru, epochs=10, batch_size=64, validation_split=0.2, verbose=2)
y_pred_gru_central = np.argmax(gru_model.predict(X_test_gru), axis=1)
print("\n--- Centralized GRU Classification Report ---")
print(classification_report(y_test, y_pred_gru_central))
acc_gru_central = accuracy_score(y_test, y_pred_gru_central)
print(f"The Accuracy using Centralized GRU method is : {acc_gru_central*100:.3f} %")

# Model 4: Gradient Boosting 

In [ ]:
gbdt = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt.fit(X_train, y_train)
y_pred_gbdt = gbdt.predict(X_test)
print("Gradient Boosting Decision Trees Classification Report:")
print(classification_report(y_test, y_pred_gbdt))

In [ ]:
acc = accuracy_score(y_test, y_pred_gbdt)
print(f"The Accuracy using Gradient Boosting is: {acc*100:.3f} %")

# Model 5: Multi-Layer Perceptron (MLP)

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=100, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
print("Multi-Layer Perceptron Classification Report:")
print(classification_report(y_test, y_pred_mlp))

In [ ]:
acc = accuracy_score(y_test, y_pred_mlp)
print(f"The Accuracy using mlp is : {acc*100:.3f} %")

# Model 6: Stacking Ensemble

In [ ]:
estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gbdt', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=100, random_state=42))
]

stacking = StackingClassifier(estimators=estimators, final_estimator=RandomForestClassifier(n_estimators=100, random_state=42))
stacking.fit(X_train, y_train)
y_pred_stacking = stacking.predict(X_test)
print("Stacking Ensemble Classification Report:")
print(classification_report(y_test, y_pred_stacking))

In [ ]:
acc = accuracy_score(y_test, y_pred_stacking)
print(f"The Accuracy using stacking ensemble method is : {acc*100:.3f} %\n")

# --- Federated Learning Simulation (GRU) ---

In [ ]:
print("\n\n--- Starting Federated Learning Simulation ---")

def create_client_data(X_train, y_train, user_series, num_clients=100):
    client_data = defaultdict(lambda: {'X': [], 'y': []})
    
    user_df = pd.DataFrame({'user': user_series, 'idx': user_series.index})
    X_df = pd.DataFrame(X_train)
    y_df = pd.DataFrame(y_train, columns=['anomaly'])

    data_df = pd.concat([
        user_df.reset_index(drop=True),
        X_df.reset_index(drop=True),
        y_df.reset_index(drop=True)
    ], axis=1)

    unique_users = user_series.unique()
    users_per_client = np.array_split(unique_users, num_clients)
    
    client_id = 0
    for user_group in users_per_client:
        if not user_group.size:
            continue
        
        client_name = f"client_{client_id + 1}"
        client_id += 1
        
        client_indices = data_df[data_df['user'].isin(user_group)].index
        
        client_X_data = X_df.iloc[client_indices].values
        client_y_data = y_df.iloc[client_indices].values
        
        if len(client_X_data) == 0:
            continue

        client_X_gru = client_X_data.reshape((client_X_data.shape[0], 1, client_X_data.shape[1]))
        client_y_cat = to_categorical(client_y_data, num_classes=2)
        
        client_data[client_name]['X'] = client_X_gru
        client_data[client_name]['y'] = client_y_cat

    print(f"Created {len(client_data)} clients.")
    return client_data


def create_gru_model(input_shape):
    model = Sequential()
    model.add(GRU(50, input_shape=input_shape, return_sequences=True))
    model.add(GRU(50))
    model.add(Dense(2, activation='softmax'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

input_shape = (X_train_gru.shape[1], X_train_gru.shape[2])
global_model = create_gru_model(input_shape)

NUM_CLIENTS = 100
clients_data = create_client_data(X_train, y_train, X_train_users, NUM_CLIENTS)
client_names = list(clients_data.keys())

COMMUNICATION_ROUNDS = 10
CLIENTS_PER_ROUND = 10
LOCAL_EPOCHS = 1
BATCH_SIZE = 32

print(f"Total Clients: {len(client_names)}")
print(f"Communication Rounds: {COMMUNICATION_ROUNDS}")
print(f"Clients per Round: {CLIENTS_PER_ROUND}")
print(f"Local Epochs: {LOCAL_EPOCHS}")

In [ ]:
for comm_round in range(COMMUNICATION_ROUNDS):
    print(f"\n--- Communication Round {comm_round + 1}/{COMMUNICATION_ROUNDS} ---")
    
    global_weights = global_model.get_weights()
    client_weights = []
    clients_in_round = np.random.choice(client_names, CLIENTS_PER_ROUND, replace=False)
    
    for client_name in clients_in_round:
        local_model = create_gru_model(input_shape)
        local_model.set_weights(global_weights)
        
        client_X = clients_data[client_name]['X']
        client_y = clients_data[client_name]['y']
        
        if len(client_X) < 1:
            print(f"Skipping {client_name}: no data.")
            continue
        
        local_model.fit(client_X, client_y, epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
        
        client_weights.append(local_model.get_weights())
        print(f"Client {client_name} trained.")

    if not client_weights:
        print("No clients trained in this round. Skipping weight aggregation.")
        continue

    new_global_weights = []
    for weights_list_tuple in zip(*client_weights):
        new_global_weights.append(np.mean(weights_list_tuple, axis=0))
        
    global_model.set_weights(new_global_weights)
    print("Global model weights updated by averaging client weights.")

print("\n--- Federated Learning Training Complete ---")

# Final Evaluation & Comparison

In [ ]:
print("\n--- Evaluating Federated GRU Model ---")
y_pred_fl = np.argmax(global_model.predict(X_test_gru), axis=1)

print("\n--- Federated GRU Classification Report ---")
print(classification_report(y_test, y_pred_fl))

acc_fl = accuracy_score(y_test, y_pred_fl)
print(f"The Accuracy using Federated GRU method is : {acc_fl*100:.3f} %")

print("\n--- Comparison ---")
print(f"Centralized GRU Accuracy: {acc_gru_central*100:.3f} %")
print(f"Federated GRU Accuracy  : {acc_fl*100:.3f} %")